<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

# Introduction to the `dts devel` API

The `dts devel` API is designed to work with a [`DTProject`](https://docs.duckietown.com/ente/duckietown-manual/70-developer-manual/dtproject/introduction-to-duckietown-projects.html#). For a full overview of building and working with `DTProjects` to create new demos
[see this section in the Duckietown manual](https://docs.duckietown.com/ente/duckietown-manual/70-developer-manual/dtproject/creating-demos.html).

The repo that we are going to use as our `DTProject` is the [`dt-core`](https://github.com/duckietown/dt-core), which you should have a copy of in the [packages](../packages/) folder if you followed the instructions about initializing the submodule in the [README](../README.md). You can notice that the `dt-core` repo has all of the elements of a `DTProject`, including:

 - the code in the [packages](../packages/dt-core/packages/) directory (inside `dt-core` - not to be confused with the `packages` directory in this learning experience)
 - the dependencies defined
 - the [launchers](../packages/dt-core/launchers/) defined

The remainder of the instructions in this part will be run from the directory [packages/dt-core](../packages/dt-core/), which is the base directory of your `DTProject`. You can move into that directory with

```bash
    cd packages/dt-core
```

## The Duckiematrix

Similar to learning experiences, we can run demos with `dts devel` on a real Duckiebot or on a virtual one inside the Duckiematrix. 

The steps to get started with the Duckiematrix are described in [this section of the book](https://docs.duckietown.com/ente/duckietown-manual/50-duckiematrix/introduction-to-the-duckiematrix-virtual-environment.html). 

In summary, the process is, depending on the environment you are using, one of the two below:

### The Duckiematrix on native Ubuntu system

If you are running Duckietown in a native Ubuntu environment, then:

1. Run the matrix with a map that is appropriate for the demo you are trying to run. In this case there is a map specified inside the project repo:

```bash
    dts matrix run --standalone --embedded -m big_loop
```

2. In a new terminal, `create`, `start`, and `attach` a virtual robot (call it anything you want - here we use `ROBOT_NAME`) to the Duckiebot asset (in this case `map_0/vehicle_0`) that is inside the Duckiematrix

```bash
    dts duckiebot virtual create --type duckiebot --configuration DB21J ROBOT_NAME
    dts duckiebot virtual start ROBOT_NAME
    dts matrix attach ROBOT_NAME map_0/vehicle_0
```

### The Duckiematrix on macOS or Windows through Duckietown Workspaces

If you are running Duckietown in a Workspace, then:

1. Inside the workspace: run the matrix engine using a map that is appropriate for the demo you are trying to run. In this case there is a map specified inside the project repo:

```bash
    dts matrix engine run --embedded -m big_loop
```

2. On the host machine, open a terminal and run:

```bash
    dts matrix run
```

2. In a new terminal inside the workspace, `create`, `start`, and `attach` a virtual robot (call it anything you want - here we use `ROBOT_NAME`) to the Duckiebot asset (in this case `map_0/vehicle_0`) that is inside the Duckiematrix:

```bash
    dts duckiebot virtual create --type duckiebot --configuration DB21J ROBOT_NAME
    dts duckiebot virtual start ROBOT_NAME
    dts matrix attach ROBOT_NAME map_0/vehicle_0
```

---

From this point on the virtual robot will be treated identically to a real robot. 


## Building the code

The next essential steps are building the code inside the `dt-core` package and running it on the Duckiebot. To build the code, make sure you are in the `packages/dt-core` directory and run:

    dts devel build -H ROBOT_NAME

where `ROBOT_NAME` is the name of your robot (real or virtual).

## Running the code

To run the code you will need to specify a launcher (or use the default one - named `default.sh` in the `launcher` directory). For the lane following demo, the launcher is called `lane-following.sh`. We can run it using the command:

    dts devel run -H ROBOT_NAME -L lane-following

You will know the command was executed successfully when the LEDs of your robot turned from white and red to green.

## Other useful tips

It can be useful to look at the command line options for both `dts devel build` and `dts devel run` by adding the `--help` option. For our purposes the following two options with `dts devel run` may be useful. 

### Forcing a build with local changes

The `-f` flag can be added to force the project to build even if there are local changes

### Running the code locally on your computer

You can build the code locally by omitting the `-H ROBOT_NAME`:

    dts devel build

and then run it and connect to your robot with 

    dts devel run -R ROBOT_NAME

### Omiting re-building for local changes that don't require recompiling

One of the advantages of Python is that it is a scripting language, so we do not always need to compile things. Given that this demo is completely implemented in Python and ROS, if the only changes you have made are modifications to parameters or code that does not require changes, there is actually no need to rerun the `dts build` command. Instead you can just sync your code and mount it in:

    dts devel run -H ROBOT_NAME -L lane-following -s -M



# Running the lane following demo

When you ran `dts devel run` you should have noticed that your LEDs turned green. This indicates that the robot is ready to start autonomous lane following mode. 

To test it you can place it in a lane (either a real one or in the Duckiematrix), and then, in a new terminal, run the `keyboard_control`:

    dts duckiebot keyboard_control ROBOT_NAME

With the keyboard control window in focus, you can press the `F` or toggle the `AutoPilot` switch. 

## Tuning parameters

You can add a new `ROBOT_NAME.yaml` file to the `config` folder in any package in the `dt-core` repository and those parameters will be taken at startup in place of the ones specified in `default.yaml`

In most cases, you can modify the parameters using the utility `rosparam set`. However, these values will not persist if you restart the demo until you change the actual values that are loaded from the `config` folder of the node.


## An overview of how it works

The `Lane Following` demo consists of the following processing steps:


1. **Image capture**: an image is captured from your Duckiebot's camera.
2. **Line detection**: colored line segments in the captured image are detected.
3. **Ground projection**: based on the camera's known intrinsic and extrinsic parameters, the detected line segments are projected onto the ground plane.
4. **Lane estimation**: the ground projected line segments and motion of your Duckiebot, which can be estimated with the help of its wheel encoders, are used to produce an estimate of your Duckiebot's lane pose.
5. **Control**: based on the lane pose estimate, a PID controller sends control signals to adjust your Duckiebot's heading.
6. **Actuation**: the control signals are applied to your Duckiebot's motors.


In the remainder of the notebooks in this learning experience, we will go through the components of the lane following demo one by one, describe how they work, and what parameters could be tuned to get the lane following behavior to work better in your environment. 

## A note on calibration

These instructions assume that you have fully [setup](https://docs.duckietown.com/ente/duckietown-manual/09-db-opmanual-intro/setup-duckiebot-intro.html) and [calibrated](https://docs.duckietown.com/ente/duckietown-manual/20-operations/04-calibrations/duckiebot-calibrations-intro.html) your Duckiebot. 


Let's proceed to the notebook [about the line detection](./01_line_detection.ipynb).

### Credits

The lane following bahavior itself is the result of work of countless people who have contributed to the `dt-core` repository. A good percentage of the text describing the components was originally written by Adam Burhan and Azalée Robitaille the Université of Montréal. 